# Fine-Tuning DistilBERT for Gmail Category Classification

**Why fine-tuning?**  
DistilBERT was pre-trained on billions of tokens across 104 languages — it already understands grammar, semantics, and word relationships. I just add a small classification head on top and nudge the weights toward my 7 Gmail categories. This needs far less data and compute than training from scratch.

In [ ]:
%pip install transformers datasets scikit-learn -q

## 1 — Imports & Config

In [ ]:
import sqlite3, pickle, os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME  = "distilbert-base-multilingual-cased"
BATCH_SIZE  = 32
MAX_LEN     = 128
EPOCHS      = 3
LR          = 2e-5

ON_COLAB = os.path.exists("/content")
DATA_FILE = "/content/emails.parquet" if ON_COLAB else "emails.parquet"
SAVE_DIR  = "/content/distilbert_model"  if ON_COLAB else "distilbert_model"

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Apple MPS")
else:
    device = torch.device("cpu")
    print("✗ CPU — consider enabling GPU in Runtime → Change runtime type")

print(f"Colab: {ON_COLAB} | Data: {DATA_FILE}")

## 2 — Load & Inspect Data

In [ ]:
# Upload emails.parquet when running on Colab
if ON_COLAB and not os.path.exists(DATA_FILE):
    from google.colab import files
    uploaded = files.upload()  # select emails.parquet from your Mac

df = pd.read_parquet(DATA_FILE)
df = df.dropna(subset=["subject", "label"])
df["text"] = df["subject"].fillna("") + " " + df["body_preview"].fillna("")

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])
num_classes = len(le.classes_)

print(f"{len(df)} emails | {num_classes} classes\n")
df["label"].value_counts().plot(kind="barh", figsize=(7, 4), color="steelblue")
plt.title("Label Distribution")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

## 3 — Train / Test Split & Tokenisation

**Why `stratify`?** With imbalanced classes, a random split could put all "forums" emails in training and none in test. Stratification keeps the class ratio identical in both splits.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].values, df["label_id"].values,
    test_size=0.2, random_state=42, stratify=df["label_id"].values,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

print(f"\nLoading tokeniser …")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class EmailDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=MAX_LEN
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

print("Tokenising train set …")
train_dataset = EmailDataset(X_train, y_train)
print("Tokenising test set …")
test_dataset  = EmailDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)
print("Done.")

Train: 15140 | Test: 3785

Loading tokeniser …
Tokenising train set …
Tokenising test set …
Done.


## 4 — Load Model & Optimiser

**Why `learning_rate=2e-5`?** Too high → catastrophic forgetting (pre-trained weights get destroyed). Too low → model barely adapts. 2e-5–5e-5 is the standard sweet spot for BERT-family fine-tuning.

**Why warmup steps?** The first ~10% of training steps use a linearly increasing LR, then it decays. This prevents large gradient updates at the start from damaging the pre-trained representations.

In [11]:
print(f"Loading {MODEL_NAME} …")
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_classes
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Model on: {device}")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading distilbert-base-multilingual-cased …


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model on: mps
Total steps: 2841 | Warmup: 284
Parameters: 135,330,055


## 5 — Training Loop

Loss should decrease each epoch. If it goes up → learning rate too high or overfitting.

In [ ]:
epoch_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss, steps = 0, 0

    for step, batch in enumerate(train_loader):
        batch    = {k: v.to(device) for k, v in batch.items()}
        outputs  = model(**batch)
        loss     = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        steps += 1

        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/steps:.4f}")

    avg_loss = total_loss / steps
    epoch_losses.append(avg_loss)
    print(f"✓ Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}\n")

# Plot loss curve
plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCHS+1), epoch_losses, marker="o", color="steelblue")
plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Avg Loss")
plt.tight_layout()
plt.show()

  Epoch 1 | Step 100/947 | Loss: 1.8092


## 6 — Evaluation

**Weighted F1** accounts for class imbalance — large classes count more.  
**Macro F1** is the unweighted average — shows how well rare classes (forums, spam) are handled.

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch   = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds   = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_, digits=3))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix — DistilBERT")
plt.tight_layout()
plt.show()

## 7 — Save Model

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f"{SAVE_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print(f"Model saved → {SAVE_DIR}/")
print(f"Files: {os.listdir(SAVE_DIR)}")